# Stakeholder Gym — BOLD dual-GPU GRPO on Kaggle T4 x2

**True dual-GPU via accelerate + DDP.** Not Unsloth (single-GPU). Uses HF transformers + bitsandbytes 4-bit + PEFT LoRA + TRL GRPOTrainer. Effective batch doubles vs single-GPU.

Settings:
- Model: **Qwen 2.5-3B-Instruct** (6× the 0.5B Colab run)
- Training: **200 steps**, LR **2e-5**, 6 generations per prompt
- Effective batch: `per_device_batch=2 × num_gpus=2 × grad_accum=4 = 16` (vs 8 single-GPU)
- Dataset: L0 + L1 + L2 prompts, ~60+ samples

Kaggle setup: **Accelerator → GPU T4 x2**. Total wall time ~60-75 min.

## 0. Verify BOTH GPUs attached — fail fast

In [ ]:
!nvidia-smi
import torch
n = torch.cuda.device_count()
assert n >= 2, f'Need 2 GPUs, got {n}. Settings → Accelerator → GPU T4 x2.'
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f'[{i}] {p.name}  {p.total_memory/1e9:.1f}GB  compute {p.major}.{p.minor}')

## 1. Deps + clone

Skip Unsloth (single-GPU). Use the canonical multi-GPU stack: transformers + bitsandbytes + peft + trl + accelerate.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets pydantic networkx pyyaml matplotlib

import os, subprocess
REPO = 'https://github.com/SAISriram19/meta.git'
WORK = '/kaggle/working/meta'
if not os.path.isdir(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True)
%cd {WORK}
!git log --oneline -1

## 2. Accelerate config — distributed mode, bf16, 2 processes

Write a minimal accelerate config instead of the interactive wizard.

In [ ]:
import os, yaml
cfg = {
    'compute_environment': 'LOCAL_MACHINE',
    'distributed_type': 'MULTI_GPU',
    'mixed_precision': 'bf16',
    'num_processes': 2,
    'num_machines': 1,
    'machine_rank': 0,
    'main_training_function': 'main',
    'gpu_ids': 'all',
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}
os.makedirs('/root/.cache/huggingface/accelerate', exist_ok=True)
with open('/root/.cache/huggingface/accelerate/default_config.yaml', 'w') as f:
    yaml.dump(cfg, f)
!accelerate env

## 3. Baseline eval — BEFORE numbers (single-GPU, fast)

In [ ]:
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled,memory_aware \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train_kaggle

import json
with open('eval_outputs/pre_train_kaggle/summary.json') as f:
    print(json.dumps(json.load(f), indent=2))

## 4. Launch distributed GRPO training

Both GPUs active. `accelerate launch --num_processes=2` replicates the model across GPUs (DDP), splits the batch. Watch `nvidia-smi` in a separate cell mid-run to confirm both GPUs at >50% utilization.

In [ ]:
!accelerate launch --num_processes=2 --mixed_precision=bf16 \
    scripts/train_grpo_ddp.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --max-steps 200 \
    --lr 2e-5 \
    --per-device-batch 2 \
    --grad-accum 4 \
    --num-generations 6 \
    --max-completion-length 200 \
    --temperature 0.9 \
    --output /kaggle/working/outputs/grpo-stakeholder

## 5. Plot reward curve from the dumped log

In [ ]:
import json, os
import matplotlib.pyplot as plt

with open('/kaggle/working/outputs/grpo-stakeholder-lora/reward_log.json') as f:
    log = json.load(f)
steps = [h['step'] for h in log]
rewards = [h['reward'] for h in log]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, label='mean reward per batch', marker='o', alpha=0.4, markersize=3)
window = 10
if len(rewards) >= window:
    ma = [sum(rewards[max(0,i-window+1):i+1])/min(i+1,window) for i in range(len(rewards))]
    plt.plot(steps, ma, label=f'{window}-step moving avg', linewidth=2.5, color='C1')
plt.xlabel('training step')
plt.ylabel('mean reward')
plt.title(f'Qwen 2.5-3B GRPO dual-GPU — {len(rewards)} steps, effective batch 16')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/grpo_reward_curve_3b_ddp.png', dpi=120)
plt.show()
print(f'first: {rewards[0]:+.4f}  last: {rewards[-1]:+.4f}  delta: {rewards[-1]-rewards[0]:+.4f}')

## 6. Post-train eval — load LoRA adapters, evaluate on L0 + L2

In [ ]:
import sys, json, torch
from pathlib import Path
sys.path.insert(0, '.')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from env.environment import StakeholderEnv
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT
from scripts.train import format_prompt, parse_completion

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
LORA_PATH = '/kaggle/working/outputs/grpo-stakeholder-lora'

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
model = PeftModel.from_pretrained(base, LORA_PATH)
model.eval()

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.4, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

config_eval = EvalConfig(
    policies={'trained_qwen3b_ddp': make_trained_policy()},
    scenarios=['L0_launch', 'L2_strategic_shift'],
    seeds=[0, 1, 2],
    out_dir=Path('eval_outputs/post_train_kaggle'),
)
run_eval(config_eval, verbose=True)

## 7. Before / After

In [ ]:
import json
with open('eval_outputs/pre_train_kaggle/summary.json') as f:
    pre = json.load(f)
with open('eval_outputs/post_train_kaggle/summary.json') as f:
    post = json.load(f)

# summary.json shape is {"cells": [...]} — unwrap.
pre_rows = pre['cells'] if isinstance(pre, dict) and 'cells' in pre else pre
post_rows = post['cells'] if isinstance(post, dict) and 'cells' in post else post

print(f'{"kind":<7} {"policy":<22} {"scenario":<22} {"reward":>8} {"bad":>5} {"terminal":>8} {"princ":>6}')
print('-' * 92)
for row in pre_rows:
    print(f'BEFORE  {row["policy"]:<22} {row["scenario"]:<22} {row["total_reward_mean"]:>+8.3f} {row["bad_agreements_mean"]:>5.1f} {row["terminal_score_mean"]:>+8.3f} {row["principled_mean"]:>6.1f}')
print()
for row in post_rows:
    print(f'AFTER   {row["policy"]:<22} {row["scenario"]:<22} {row["total_reward_mean"]:>+8.3f} {row["bad_agreements_mean"]:>5.1f} {row["terminal_score_mean"]:>+8.3f} {row["principled_mean"]:>6.1f}')

## 8. Package artifacts for download

In [ ]:
import shutil, tarfile, os
out = '/kaggle/working/outputs'
shutil.copy('eval_outputs/pre_train_kaggle/summary.json', f'{out}/pre_train_summary.json')
shutil.copy('eval_outputs/post_train_kaggle/summary.json', f'{out}/post_train_summary.json')
with tarfile.open(f'{out}/grpo-stakeholder-lora.tar.gz', 'w:gz') as t:
    t.add(f'{out}/grpo-stakeholder-lora', arcname='grpo-stakeholder-lora')
for f in sorted(os.listdir(out)):
    sz = os.path.getsize(f'{out}/{f}') if os.path.isfile(f'{out}/{f}') else '-'
    print(f'  {f}  ({sz})')